# Quantum Sun Playground

This notebook demonstrates the `Polfed.QSun` model constructors. Edit the parameters below, then rerun the cells to compare the full Hilbert-space Hamiltonian with the U(1)-conserving fixed-`S_z` sector.

In [ ]:
using LinearAlgebra
using Printf
using Random
using SparseArrays

using Polfed.QSun: qsun_hamiltonian

## Parameters

For quick experimentation, keep the system small. The full spin-1/2 Hilbert-space dimension is `2^(L_loc + L_grain)`, while the U(1) sector is smaller.

In [ ]:
L_loc = 4
L_grain = 2
g0 = 1.0
alpha = 0.55

S = 0.5
gamma = 1.0
w = 0.5
hz = 1.0
zeta = 0.2
sector_Sz = 0.0
seed = 1234;

In [ ]:
function summarize_matrix(name::AbstractString, H)
    Hdense = Matrix(H)
    eigs = eigvals(Hermitian(Hdense))

    println("\n", name)
    println(repeat("-", length(name)))
    println("size       = ", size(H))
    println("nnz        = ", H isa SparseMatrixCSC ? nnz(H) : "dense")
    println("density    = ", @sprintf("%.3f", count(!iszero, Hdense) / length(Hdense)))
    println("Hermitian  = ", ishermitian(Hdense))
    println("spectrum   = [", @sprintf("%.6f", minimum(eigs)), ", ", @sprintf("%.6f", maximum(eigs)), "]")
    println("first eigs = ", eigs[1:min(5, end)])

    return eigs
end;

## Full Hilbert Space

In [ ]:
H_full = qsun_hamiltonian(
    L_loc,
    L_grain,
    g0,
    alpha;
    S=S,
    γ=gamma,
    w=w,
    hz=hz,
    ζ=zeta,
    rng=MersenneTwister(seed),
    use_sparse=true,
)

eigs_full = summarize_matrix("Full Hilbert-space QSun", H_full);

## U(1)-Conserving Sector

The U(1) version preserves total magnetization and builds only the chosen sector. For spin-1/2 with an even number of sites, `S_z = 0` is the central sector.

In [ ]:
H_u1 = qsun_hamiltonian(
    L_loc,
    L_grain,
    g0,
    alpha;
    S=S,
    γ=gamma,
    w=w,
    hz=hz,
    ζ=zeta,
    rng=MersenneTwister(seed),
    use_U1=true,
    S_z=sector_Sz,
    use_sparse=true,
)

eigs_u1 = summarize_matrix("U(1)-conserving S_z=$(sector_Sz) sector", H_u1);

## Try Next

- Change `L_loc`, `L_grain`, `alpha`, or `sector_Sz` and rerun.
- Use `use_sparse=false` only for very small systems where dense exact diagonalization is practical.
- For larger sparse runs, pass `H_full` or `H_u1` into `Polfed.polfed` with the usual POLFED configuration.